# 03 - Embedding and FAISS index: model benchmark

**Project:** AI-Powered Chatbot for Data Engineering Students (FYP, UiTM)
**Phase 6, Step 3:** turn every chunk into a vector, store the vectors in FAISS, and choose the embedding model.

## What this notebook does
1. Loads the 2,515 labelled chunks (`data/chunks/chunks.json`) and the 100 approved test questions (`evaluation/test_questions.json`).
2. Embeds all chunks with **both candidate models** from `settings.EMBEDDING_CANDIDATES`:
   - `all-MiniLM-L6-v2` (the default in the proposal)
   - `BAAI/bge-small-en-v1.5` (candidate)
3. Builds one FAISS index per model.
4. Asks every test question against every index and measures whether a correct chunk comes back:
   - **hit@k**: share of questions where at least one relevant chunk is in the top k results (k = 1, 3, 5, 10)
   - **MRR** (mean reciprocal rank): 1 if the first relevant chunk is ranked 1st, 1/2 if 2nd, 1/3 if 3rd ... 0 if not in the top 10. It rewards ranking the right chunk high.
5. Compares the models overall, per question, per topic and per source type.

Both models run **locally and free** (sentence-transformers); no API is used.

## Limitations of the test set (see `evaluation/test_questions_limitations.md`)
- n = 100: one question is 1 percentage point of hit@k, and a hit rate around 0.75 carries roughly ±8.5 points of uncertainty (95%). Small differences of a few questions are still within noise; the head-to-head count is the more sensitive comparison.
- Q06, Q11, Q59 and Q77 lean on literal identifier matching, so results are also reported **without** them.

**Status: DECIDED.** `BAAI/bge-small-en-v1.5`, used without a query prefix, is the embedding model of the chatbot (section 5). Its FAISS index is saved to `data/faiss_index/` (section 6).

## 1. Setup

In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import settings

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

chunks = json.loads(settings.CHUNKS_FILE.read_text(encoding="utf-8"))
test_set = json.loads(settings.TEST_QUESTIONS_FILE.read_text(encoding="utf-8"))
questions = test_set["questions"]
literal_ids = set(test_set["meta"]["literal_identifier_questions"])
chunk_ids = [c["chunk_id"] for c in chunks]

# Every relevant chunk id in the test set must exist in the current chunks.json
known = set(chunk_ids)
missing = [(q["id"], cid) for q in questions for cid in q["relevant_chunk_ids"] if cid not in known]
print(f"Chunks: {len(chunks)} | test questions: {len(questions)} | relevant chunk ids: "
      f"{sum(len(q['relevant_chunk_ids']) for q in questions)} | unknown ids: {len(missing)} (should be 0)")
print("Test set status:", test_set["meta"]["status"])
print("Candidates:", settings.EMBEDDING_CANDIDATES)

Chunks: 2515 | test questions: 100 | relevant chunk ids: 329 | unknown ids: 0 (should be 0)
Test set status: APPROVED - all 100 questions frozen (Q01-Q28 approved 2026-10-13, Q29-Q100 approved 2026-10-19)
Candidates: ['sentence-transformers/all-MiniLM-L6-v2', 'BAAI/bge-small-en-v1.5']


## 2. What is embedded, and does it fit the model?
Each chunk is embedded as `settings.EMBED_TEXT_TEMPLATE` = section label + text, so the heading gives context.

Each model reads at most a fixed number of **tokens** (word pieces); anything longer is cut off silently. The check below counts how many chunks are longer than each model's limit.

In [2]:
texts = [settings.EMBED_TEXT_TEMPLATE.format(section=c["section"], text=c["text"]) for c in chunks]

models = {name: SentenceTransformer(name) for name in settings.EMBEDDING_CANDIDATES}

rows = []
for name, model in models.items():
    lengths = [len(ids) for ids in model.tokenizer(texts, add_special_tokens=True)["input_ids"]]
    limit = model.max_seq_length
    over = [n for n in lengths if n > limit]
    rows.append({"model": name, "max tokens": limit, "dimensions": model.get_sentence_embedding_dimension(),
                 "median chunk tokens": int(np.median(lengths)), "max chunk tokens": max(lengths),
                 "chunks truncated": len(over), "% truncated": round(100 * len(over) / len(lengths), 1)})
print(pd.DataFrame(rows).to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


C:\Users\thaf\AppData\Local\Temp\ipykernel_11232\442343726.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  rows.append({"model": name, "max tokens": limit, "dimensions": model.get_sentence_embedding_dimension(),


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (560 > 512). Running this sequence through the model will result in indexing errors


                                 model  max tokens  dimensions  median chunk tokens  max chunk tokens  chunks truncated  % truncated
sentence-transformers/all-MiniLM-L6-v2         256         384                  136               560               116          4.6
                BAAI/bge-small-en-v1.5         512         384                  136               560                 3          0.1


## 3. Embed all chunks and build one FAISS index per model
Vectors are **normalised** (length 1), so the inner product used by `faiss.IndexFlatIP` equals cosine similarity. `IndexFlatIP` is an exact search: every query is compared with every chunk, which is fast enough for 2,515 chunks and gives exact results (no approximation to tune).

In [3]:
indexes, timings = {}, {}
for name, model in models.items():
    start = time.perf_counter()
    vectors = model.encode(texts, batch_size=settings.EMBEDDING_BATCH_SIZE, normalize_embeddings=True,
                           show_progress_bar=False, convert_to_numpy=True).astype("float32")
    timings[name] = time.perf_counter() - start
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    indexes[name] = index
    print(f"{name:42s} {vectors.shape[0]} vectors x {vectors.shape[1]} dims | "
          f"embedding time {timings[name]:.0f} s | index size {vectors.nbytes / 1e6:.1f} MB")

sentence-transformers/all-MiniLM-L6-v2     2515 vectors x 384 dims | embedding time 70 s | index size 3.9 MB


BAAI/bge-small-en-v1.5                     2515 vectors x 384 dims | embedding time 171 s | index size 3.9 MB


## 4. Benchmark
Each configuration = a model plus how the **question** is encoded. bge-small-en-v1.5 is trained to expect an instruction in front of short search queries (`settings.EMBEDDING_QUERY_PREFIX`), so it is measured **with and without** that prefix. The chunks themselves are never prefixed.

For every question the top 10 chunks are retrieved; the **rank** is the position of the first relevant chunk (None if none is in the top 10).

In [4]:
TOP_N = max(settings.BENCHMARK_TOP_K)

configs = []
for name in settings.EMBEDDING_CANDIDATES:
    short = name.split("/")[-1]
    prefix = settings.EMBEDDING_QUERY_PREFIX.get(name)
    if prefix:
        configs.append({"label": f"{short} (no prefix)", "model": name, "prefix": ""})
        configs.append({"label": f"{short} + query prefix", "model": name, "prefix": prefix})
    else:
        configs.append({"label": short, "model": name, "prefix": ""})


def run_benchmark(config):
    """Retrieve the top TOP_N chunks for every question; return per-question rank and retrieved ids."""
    model = models[config["model"]]
    query_vectors = model.encode([config["prefix"] + q["question"] for q in questions],
                                 normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, positions = indexes[config["model"]].search(query_vectors, TOP_N)
    results = []
    for q, row, score_row in zip(questions, positions, scores):
        retrieved = [chunk_ids[i] for i in row]
        relevant = set(q["relevant_chunk_ids"])
        rank = next((r for r, cid in enumerate(retrieved, start=1) if cid in relevant), None)
        results.append({"id": q["id"], "rank": rank, "retrieved": retrieved, "top_score": float(score_row[0])})
    return results


def metrics(results):
    """hit@k for every k in BENCHMARK_TOP_K and MRR over the top TOP_N."""
    n = len(results)
    out = {f"hit@{k}": sum(r["rank"] is not None and r["rank"] <= k for r in results) / n for k in settings.BENCHMARK_TOP_K}
    out["MRR"] = sum(1 / r["rank"] for r in results if r["rank"] is not None) / n
    out["missed (not in top 10)"] = sum(r["rank"] is None for r in results)
    return out


benchmark = {c["label"]: run_benchmark(c) for c in configs}
print("Configurations:", [c["label"] for c in configs])

Configurations: ['all-MiniLM-L6-v2', 'bge-small-en-v1.5 (no prefix)', 'bge-small-en-v1.5 + query prefix']


### 4.1 Overall results

In [5]:
def results_table(subset_ids=None):
    """One row per configuration; optionally only for a subset of question ids."""
    rows = []
    for label, results in benchmark.items():
        chosen = [r for r in results if subset_ids is None or r["id"] in subset_ids]
        rows.append({"configuration": label, **metrics(chosen)})
    table = pd.DataFrame(rows).set_index("configuration")
    for col in table.columns:
        if col.startswith("hit@") or col == "MRR":
            table[col] = table[col].map(lambda v: f"{v:.3f}")
    return table


all_ids = {q["id"] for q in questions}
print(f"ALL {len(all_ids)} QUESTIONS")
print(results_table().to_string())
without_literal = all_ids - literal_ids
print(f"\nWITHOUT the literal-identifier questions {sorted(literal_ids)} ({len(without_literal)} questions)")
print(results_table(without_literal).to_string())

ALL 100 QUESTIONS
                                  hit@1  hit@3  hit@5 hit@10    MRR  missed (not in top 10)
configuration                                                                              
all-MiniLM-L6-v2                  0.710  0.900  0.970  0.970  0.815                       3
bge-small-en-v1.5 (no prefix)     0.830  0.950  0.970  0.980  0.892                       2
bge-small-en-v1.5 + query prefix  0.810  0.940  0.960  0.980  0.876                       2

WITHOUT the literal-identifier questions ['Q06', 'Q11', 'Q59', 'Q77'] (96 questions)
                                  hit@1  hit@3  hit@5 hit@10    MRR  missed (not in top 10)
configuration                                                                              
all-MiniLM-L6-v2                  0.719  0.906  0.969  0.969  0.820                       3
bge-small-en-v1.5 (no prefix)     0.844  0.958  0.969  0.979  0.901                       2
bge-small-en-v1.5 + query prefix  0.823  0.938  0.958  0.979  0.883 

### 4.2 Rank of the first relevant chunk, per question

In [6]:
per_question = pd.DataFrame({
    "topic": [q["topic"] for q in questions],
    "seed type": [q["seed_source_type"] for q in questions],
    **{label: [r["rank"] if r["rank"] is not None else "-" for r in results] for label, results in benchmark.items()},
    "question": [q["question"][:62] for q in questions],
}, index=[q["id"] for q in questions])
print("Rank of the first relevant chunk (1 = best, '-' = not in the top 10)\n")
print(per_question.to_string())

Rank of the first relevant chunk (1 = best, '-' = not in the top 10)

                 topic       seed type all-MiniLM-L6-v2 bge-small-en-v1.5 (no prefix) bge-small-en-v1.5 + query prefix                                                        question
Q01   Data Warehousing    course_notes                1                             1                                1         What is the difference between OLTP and OLAP databases?
Q02   Data Warehousing        textbook                3                             1                                2             How is a data lake different from a data warehouse?
Q03   Data Warehousing    course_notes                1                             1                                1                      What are fact tables and dimension tables?
Q04   Data Warehousing    course_notes                -                             1                                1    Why is BigQuery called serverless, and why does that matter?
Q05   Data Ware

### 4.3 Head-to-head
For each pair of configurations: on how many questions does one rank a correct chunk higher than the other? (A question missed by both counts as a tie.) With n = 28, a margin of one or two questions is within noise.

In [7]:
def rank_value(rank):
    """Missing (None) counts as worse than any rank in the top 10."""
    return rank if rank is not None else TOP_N + 1


labels = list(benchmark)
rows = []
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        a, b = labels[i], labels[j]
        ra = [rank_value(r["rank"]) for r in benchmark[a]]
        rb = [rank_value(r["rank"]) for r in benchmark[b]]
        rows.append({"A": a, "B": b, "A better": sum(x < y for x, y in zip(ra, rb)),
                     "B better": sum(x > y for x, y in zip(ra, rb)), "tie": sum(x == y for x, y in zip(ra, rb))})
print(pd.DataFrame(rows).to_string(index=False))

                            A                                B  A better  B better  tie
             all-MiniLM-L6-v2    bge-small-en-v1.5 (no prefix)         7        19   74
             all-MiniLM-L6-v2 bge-small-en-v1.5 + query prefix        10        20   70
bge-small-en-v1.5 (no prefix) bge-small-en-v1.5 + query prefix         8         2   90


### 4.4 Breakdown by topic and by source type of the question (hit@5)

In [8]:
def hit5_by(field):
    """hit@5 per value of a question field (topic or seed source type), for every configuration."""
    table = {}
    for label, results in benchmark.items():
        by_id = {r["id"]: r for r in results}
        groups = {}
        for q in questions:
            groups.setdefault(q[field], []).append(by_id[q["id"]]["rank"])
        table[label] = {g: f"{sum(r is not None and r <= 5 for r in ranks)}/{len(ranks)}" for g, ranks in groups.items()}
    return pd.DataFrame(table)


print("hit@5 by topic (correct / questions)\n")
print(hit5_by("topic").to_string())
print("\nhit@5 by source type of the seed chunk\n")
print(hit5_by("seed_source_type").to_string())

hit@5 by topic (correct / questions)

                 all-MiniLM-L6-v2 bge-small-en-v1.5 (no prefix) bge-small-en-v1.5 + query prefix
Data Warehousing            20/21                         21/21                            21/21
Data Processing             18/20                         18/20                            18/20
SQL                         18/18                         18/18                            18/18
Data Pipelines              13/13                         12/13                            12/13
Data Cleaning                 9/9                           9/9                              9/9
Big Data                      8/8                           8/8                              8/8
Data Quality                  7/7                           7/7                              6/7
ETL/ELT                       4/4                           4/4                              4/4

hit@5 by source type of the seed chunk

               all-MiniLM-L6-v2 bge-small-en-v1.

### 4.5 Misses: what came back instead?
Every question that at least one configuration did **not** answer within the top 5, with the top 3 chunks each configuration returned. This shows whether a miss is a real failure or a reasonable alternative chunk that is just not on the relevant list.

In [9]:
by_id = {c["chunk_id"]: c for c in chunks}
for q in questions:
    ranks = {label: next(r for r in results if r["id"] == q["id"]) for label, results in benchmark.items()}
    if all(r["rank"] is not None and r["rank"] <= 5 for r in ranks.values()):
        continue
    print("=" * 110)
    print(f"{q['id']} [{q['topic']}] {q['question']}")
    print(f"   relevant: {', '.join(q['relevant_chunk_ids'])}")
    for label, r in ranks.items():
        print(f"   {label:34s} rank {r['rank'] if r['rank'] else '-'}")
        for cid in r["retrieved"][:3]:
            mark = "*" if cid in q["relevant_chunk_ids"] else " "
            print(f"      {mark} {cid}  {by_id[cid]['section'][:70]}")

Q04 [Data Warehousing] Why is BigQuery called serverless, and why does that matter?
   relevant: S06-0209
   all-MiniLM-L6-v2                   rank -
        S06-0210  03-data-warehouse > Data Warehouse and BigQuery > BigQuery
        S06-0250  03-data-warehouse > Internals of BigQuery
        S06-0257  03-data-warehouse > Internals of BigQuery > Dremel: executing a query 
   bge-small-en-v1.5 (no prefix)      rank 1
      * S06-0209  03-data-warehouse > Data Warehouse and BigQuery > BigQuery
        S06-0210  03-data-warehouse > Data Warehouse and BigQuery > BigQuery
        S06-0216  03-data-warehouse > Data Warehouse and BigQuery > External tables
   bge-small-en-v1.5 + query prefix   rank 1
      * S06-0209  03-data-warehouse > Data Warehouse and BigQuery > BigQuery
        S06-0210  03-data-warehouse > Data Warehouse and BigQuery > BigQuery
        S06-0257  03-data-warehouse > Internals of BigQuery > Dremel: executing a query 
Q56 [Data Processing] How does the Spark SQL job scr

## 5. Decision: bge-small-en-v1.5, without a query prefix

Results on all **100** approved test questions:

| | hit@1 | hit@3 | hit@5 | hit@10 | MRR | missed (not in top 10) | chunks truncated |
|---|---|---|---|---|---|---|---|
| all-MiniLM-L6-v2 | 0.710 | 0.900 | 0.970 | 0.970 | 0.815 | 3 | 116 (4.6%) |
| **bge-small-en-v1.5 (no prefix)** | **0.830** | **0.950** | 0.970 | **0.980** | **0.892** | **2** | **3 (0.1%)** |
| bge-small-en-v1.5 + query prefix | 0.810 | 0.940 | 0.960 | 0.980 | 0.876 | 2 | 3 (0.1%) |

Without the four literal-identifier questions (Q06, Q11, Q59, Q77; 96 questions) the order is the same: hit@1 0.719 / 0.844 / 0.823 and MRR 0.820 / 0.901 / 0.883.

**Why bge-small-en-v1.5 was chosen over all-MiniLM-L6-v2:**
1. **It ranks the right chunk first much more often.** hit@1 is 12 percentage points higher (0.830 vs 0.710) and MRR is 0.892 vs 0.815. Head to head, bge ranks a correct chunk higher on 19 questions and MiniLM on 7 (74 ties); a sign test on these 26 decided questions gives p = 0.03, so the advantage is unlikely to be noise. Ranking matters because the chatbot passes only the top chunks to the LLM, and the first ones carry the most weight.
2. **Both find a relevant chunk in the top 5 equally often** (hit@5 0.970 for both). The models differ in *how high* they rank the right chunk, not in whether it appears in the top 5. Their misses differ: MiniLM misses Q04, Q56 and Q58 entirely; bge misses Q58 and Q79 (MiniLM ranks Q79 second).
3. **It reads (almost) every chunk in full.** MiniLM reads at most 256 tokens and silently cuts off the end of 116 chunks (4.6%); bge-small reads up to 512 tokens and truncates only 3. This does not depend on the test set.
4. **Same cost at question time.** Both produce 384-dimension vectors and a 3.9 MB index; bge is only slower in the one-off embedding of all chunks (about 2 minutes instead of 1).

**Why no query prefix:** the instruction recommended for bge ("Represent this sentence for searching relevant passages: ") scored slightly lower again (no prefix ranked better on 8 questions, the prefix on 2, 90 ties; p = 0.11), so the simpler option is kept.

**Consistency with the first benchmark (n = 28):** the original 28 questions gave the same direction (hit@1 0.750 vs 0.643, MRR 0.858 vs 0.772). The larger set confirms the decision; the only change is that hit@5, which bge led at n = 28 (1.000 vs 0.964), is now tied.

**Change from the proposal:** the proposal names Sentence-BERT / all-MiniLM-L6-v2 for embeddings (Tables 2.3 and 3.9). The final report should cite this benchmark as the justification for using bge-small-en-v1.5 instead.

The decision is recorded in `config/settings.py` (`EMBEDDING_MODEL`, `EMBEDDING_QUERY_PREFIX_IN_USE`).

In [10]:
# Confirm that settings.py now points to the chosen configuration
print("EMBEDDING_MODEL               :", settings.EMBEDDING_MODEL)
print("EMBEDDING_QUERY_PREFIX_IN_USE :", repr(settings.EMBEDDING_QUERY_PREFIX_IN_USE))
assert settings.EMBEDDING_MODEL == "BAAI/bge-small-en-v1.5" and settings.EMBEDDING_QUERY_PREFIX_IN_USE == ""

EMBEDDING_MODEL               : BAAI/bge-small-en-v1.5
EMBEDDING_QUERY_PREFIX_IN_USE : ''


## 6. Save the chosen index
The bge-small index built in section 3 is written to `data/faiss_index/` (Git ignored), together with the **chunk-id order**: row *i* of the index is the vector of chunk *i* in that list. `src/core/retriever.py` will load both, so a search result (a row number) can be turned back into a chunk and its metadata. A small header records which model built the index and from how many chunks, so a stale index can be detected later.

In [11]:
settings.FAISS_INDEX_DIR.mkdir(parents=True, exist_ok=True)
chosen_index = indexes[settings.EMBEDDING_MODEL]
faiss.write_index(chosen_index, str(settings.FAISS_INDEX_FILE))

id_map = {
    "embedding_model": settings.EMBEDDING_MODEL,
    "query_prefix": settings.EMBEDDING_QUERY_PREFIX_IN_USE,
    "embed_text_template": settings.EMBED_TEXT_TEMPLATE,
    "normalized": True,                    # inner product = cosine similarity
    "index_type": "IndexFlatIP",
    "dimensions": chosen_index.d,
    "chunk_count": len(chunk_ids),
    "chunks_file": str(settings.CHUNKS_FILE.relative_to(settings.PROJECT_ROOT)),
    "chunk_ids": chunk_ids,                # row i of the index = chunk_ids[i]
}
settings.FAISS_ID_MAP_FILE.write_text(json.dumps(id_map, indent=1), encoding="utf-8")

for path in (settings.FAISS_INDEX_FILE, settings.FAISS_ID_MAP_FILE):
    print(f"Written: {path.relative_to(settings.PROJECT_ROOT)}  ({path.stat().st_size / 1e6:.2f} MB)")

Written: data\faiss_index\course_index.faiss  (3.86 MB)
Written: data\faiss_index\course_index_chunk_ids.json  (0.04 MB)


### 6.1 Check: reload the saved index and search with it
The index is read back from disk and one test question is asked, exactly as the chatbot will do it: same model, no prefix. The saved index must give the same top results as the one in memory.

In [12]:
loaded_index = faiss.read_index(str(settings.FAISS_INDEX_FILE))
loaded_map = json.loads(settings.FAISS_ID_MAP_FILE.read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_map["chunk_ids"]) == len(chunks)
assert loaded_map["chunk_ids"] == chunk_ids

question = next(q for q in questions if q["id"] == "Q28")
query = models[settings.EMBEDDING_MODEL].encode([settings.EMBEDDING_QUERY_PREFIX_IN_USE + question["question"]],
                                                normalize_embeddings=True).astype("float32")
scores, rows = loaded_index.search(query, settings.TOP_K)
_, rows_memory = chosen_index.search(query, settings.TOP_K)
assert (rows == rows_memory).all(), "saved index differs from the one in memory"

print(f"Saved index: {loaded_index.ntotal} vectors x {loaded_index.d} dims, model {loaded_map['embedding_model']}")
print(f"\nQuestion: {question['question']}\n")
for score, row in zip(scores[0], rows[0]):
    cid = loaded_map["chunk_ids"][row]
    mark = "*" if cid in question["relevant_chunk_ids"] else " "
    print(f"  {mark} {score:.3f}  {cid:9s}  {by_id[cid]['section'][:70]}")
print("\n* = on the relevant list for this question. The saved index matches the in-memory index.")

Saved index: 2515 vectors x 384 dims, model BAAI/bge-small-en-v1.5

Question: What is the difference between ETL and ELT?

  * 0.809  S06-0155   02-workflow-orchestration > ETL vs ELT
  * 0.796  S06-0322   04-analytics-engineering > Analytics Engineering Basics > ETL vs ELT —
    0.753  S06-0158   02-workflow-orchestration > ETL vs ELT > Resources
  * 0.734  S06-0156   02-workflow-orchestration > ETL vs ELT
  * 0.726  S05-0114   Data Pipeline > Data Pipeline Examples > ELT Pipeline Example

* = on the relevant list for this question. The saved index matches the in-memory index.
